In [11]:
# Core Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Model Selection and Pipelines
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Preprocessing & Feature Engineering
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures, FunctionTransformer
from sklearn.impute import SimpleImputer

# Algorithms (Built-in to sklearn)
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor

# Evaluation Metrics (Regression focused)
from sklearn.metrics import (
    mean_absolute_error,    # MAE
    mean_squared_error,     # MSE
    root_mean_squared_error, # RMSE (Note: in newer sklearn versions)
    r2_score                # R-squared
)

# Utils
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme(style="whitegrid")
%matplotlib inline

In [4]:
df = pd.read_csv('../data/vehicles.csv')
df.sample(5)

,id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,...,size,type,paint_color,image_url,description,county,state,lat,long,posting_date
338298,7305613827,https://pittsburgh.craigslist.org/ctd/d/uniont...,pittsburgh,https://pittsburgh.craigslist.org,12500,2017.0,jeep,compass,NaN,NaN,...,NaN,NaN,NaN,https://images.craigslist.org/00j0j_ie96OsAyIc...,Must see one of the cleanest Jeeps around abo...,NaN,pa,39.889700,-79.728200,2021-04-12T10:53:44-0400
98093,7316306483,https://jacksonville.craigslist.org/cto/d/lake...,jacksonville,https://jacksonville.craigslist.org,2950,2002.0,ford,taurus ses,NaN,6 cylinders,...,NaN,NaN,blue,https://images.craigslist.org/00W0W_2ngBDnAjkG...,"2002 Ford Taurus SES, 4D Sedan, 3.0L V6 Automa...",NaN,fl,30.025223,-82.350266,2021-05-03T13:44:24-0400
179952,7309962681,https://maine.craigslist.org/ctd/d/milford-201...,maine,https://maine.craigslist.org,4990,2012.0,nissan,rogue awd,NaN,4 cylinders,...,NaN,SUV,red,https://images.craigslist.org/00K0K_lPb4cBogzr...,This clean 2012 Nissan Rogue ALL WHEEL DRIVE r...,NaN,me,42.828500,-71.660600,2021-04-20T14:59:34-0400
58770,7315995099,https://santabarbara.craigslist.org/cto/d/gole...,santa barbara,https://santabarbara.craigslist.org,5999,2003.0,toyota,avalon,good,6 cylinders,...,NaN,NaN,NaN,https://images.craigslist.org/01212_67yDGjHXXD...,2003 Toyota Avalon XL low miles for the year a...,NaN,ca,34.429096,-119.835777,2021-05-02T16:42:15-0700
334569,7310365949,https://philadelphia.craigslist.org/ctd/d/spot...,philadelphia,https://philadelphia.craigslist.org,6800,2011.0,jeep,compass,NaN,4 cylinders,...,NaN,SUV,white,https://images.craigslist.org/00I0I_224JvWw097...,"2011 Jeep Compass Sport 4dr SUV -- $6,800 ...",NaN,pa,40.408250,-74.372882,2021-04-21T12:04:33-0400


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 426880 entries, 0 to 426879
Data columns (total 26 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            426880 non-null  int64  
 1   url           426880 non-null  object 
 2   region        426880 non-null  object 
 3   region_url    426880 non-null  object 
 4   price         426880 non-null  int64  
 5   year          425675 non-null  float64
 6   manufacturer  409234 non-null  object 
 7   model         421603 non-null  object 
 8   condition     252776 non-null  object 
 9   cylinders     249202 non-null  object 
 10  fuel          423867 non-null  object 
 11  odometer      422480 non-null  float64
 12  title_status  418638 non-null  object 
 13  transmission  424324 non-null  object 
 14  VIN           265838 non-null  object 
 15  drive         296313 non-null  object 
 16  size          120519 non-null  object 
 17  type          334022 non-null  object 
 18  pain

In [12]:
# id column is not useful and it is unique for each row
# url, region_url, image_url, description are not useful, description is unique long text for each row
#VIN is unique for each car, and it has a lot of missing values, so it is not useful 
# county has 0 non-null values, so it is not useful
# dropping these immidiately
initial_drop_cols = ['id','url', 'region_url', 'image_url', 'description', 'county', 'VIN']
df.drop(columns=initial_drop_cols, inplace=True)

In [6]:
df['region'].value_counts()

region
columbus                   3608
jacksonville               3562
spokane / coeur d'alene    2988
eugene                     2985
fresno / madera            2983
                           ... 
meridian                     28
southwest MS                 14
kansas city                  11
fort smith, AR                9
west virginia (old)           8
Name: count, Length: 404, dtype: int64

In [10]:
print(df['state'].nunique())
df['state'].value_counts()

51


state
ca    50614
fl    28511
tx    22945
ny    19386
oh    17696
or    17104
mi    16900
nc    15277
wa    13861
pa    13753
wi    11398
co    11088
tn    11066
va    10732
il    10387
nj     9742
id     8961
az     8679
ia     8632
ma     8174
mn     7716
ga     7003
ok     6792
sc     6327
mt     6294
ks     6209
in     5704
ct     5188
al     4955
md     4778
nm     4425
mo     4293
ky     4149
ar     4038
ak     3474
la     3196
nv     3194
nh     2981
dc     2970
me     2966
hi     2964
vt     2513
ri     2320
sd     1302
ut     1150
wv     1052
ne     1036
ms     1016
de      949
wy      610
nd      410
Name: count, dtype: int64

In [ ]:
# loaded the data, 426880 rows
# condition will be filled with 'not_reported' because it has almost 45  percent missing values, and it is a categorical variable.
# cylinders will be filled with the most frequent value for that particular car brand/model. so group by model and take mode
# will use functiontransformer in my pipeline for cylinder and age to fill the missing values.
# will probably reduce the number of states by grouping them into regions, because there are 51 unique values and it is a categorical variable., about 10 regions before one hot encoding.
# will drop the original region column cuz too many unique values and it is a categorical variable.
# will calculate the age of the car and drop the year posted column which i will get from the date posted column.
# will input the mileage with the median/mean value based on model/brand and the age
# might add poly features for age and mileage
